# Chạy SAM3 trên Google Colab

Notebook này chạy script `scripts/sam3_semantic.py` của repo và tạo ZIP semantic để import vào CVAT.

**Các task chạy được:** `cp3_thin`, `cp4_curb`, `cp6_coverage`. Bạn đã làm `easy_semantic` thì đổi biến `TASK` ở ô chọn task.

**Lưu ý:** script hiện tại chưa tạo được output instance COCO cho `medium_instance`, `cp1_holes`, `cp2_slice`, `cp5_occlusion`, và chưa tạo panoptic output cho `hard_panoptic`.

In [ ]:
# Chọn Runtime > Change runtime type > T4 GPU trước khi chạy ô này
!nvidia-smi

In [ ]:
# Dán URL fork GitHub của bạn vào đây.
# Ví dụ: https://github.com/ten-ban/ten-repo.git
REPO_URL = ""
REPO_DIR = "/content/image-segmentation"

import os
import subprocess
from pathlib import Path

if REPO_URL.strip():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

assert Path(REPO_DIR).is_dir(), (
    f"Không tìm thấy repo tại {REPO_DIR}. Hãy điền REPO_URL rồi chạy lại ô này."
)
%cd $REPO_DIR
print("Repo:", Path.cwd())

In [ ]:
# Cài thư viện SAM3
!pip install -q -U ultralytics numpy pillow

In [ ]:
# Upload checkpoint sam3.pt từ máy lên Colab
from google.colab import files
from pathlib import Path

existing = list(Path("/content").rglob("sam3.pt"))
if existing:
    MODEL = existing[0]
    print("Đã tìm thấy checkpoint:", MODEL)
else:
    uploaded = files.upload()
    names = [name for name in uploaded if name.endswith(".pt")]
    assert names, "Hãy upload đúng file sam3.pt"
    MODEL = Path("/content") / names[0]

print("Checkpoint:", MODEL)

In [ ]:
# Chọn task semantic cần chạy
TASK = "easy_semantic"

TASK_DIRS = {
    "easy_semantic": "data/tiers/easy_semantic",
    "cp3_thin": "data/checkpoints/cp3_thin",
    "cp4_curb": "data/checkpoints/cp4_curb",
    "cp6_coverage": "data/checkpoints/cp6_coverage",
}

assert TASK in TASK_DIRS, (
    f"Task không hợp lệ. Chọn một trong: {list(TASK_DIRS)}"
)
task_dir = Path(TASK_DIRS[TASK])
images = task_dir / "images"
classes = task_dir / "classes.json"
output = Path("submissions") / f"{TASK}.zip"

assert images.is_dir(), f"Thiếu thư mục: {images}"
assert classes.is_file(), f"Thiếu file: {classes}"
print("Task:", TASK)
print("Classes:", classes.read_text())

In [ ]:
# Chạy SAM3 và tạo ZIP semantic
!python scripts/sam3_semantic.py \
    --model "$MODEL" \
    --images "$images" \
    --classes "$classes" \
    --output "$output" \
    --device 0 \
    --conf 0.25

In [ ]:
# Kiểm tra ZIP trước khi tải xuống hoặc import vào CVAT
!python scripts/inspect_submissions.py --task "$TASK" --dir submissions

In [ ]:
# Tải ZIP kết quả về máy
from google.colab import files
files.download(str(output))